#  GenAI-Powered Kitchen Assistant: Capstone Notebook

**Use Case: Intelligent Meal Planner & Recipe Generator**

In today’s fast-paced world, people often struggle with deciding what to cook using the ingredients they already have. This project introduces an innovative GenAI-powered Kitchen Assistant that accepts a user's available ingredients, suggests creative recipes, and provides step-by-step cooking instructions with nutritional insights.

By combining multiple Generative AI capabilities, this assistant goes beyond static recipe apps. It can analyze images of food, retrieve relevant meals from a vector store, and evaluate recipes from a nutritionist's perspective.

# GenAI Capabilities Demonstrated

**Structured output / JSON mode**

Recipes are generated in strict JSON format, including fields like title, ingredients, steps, cooking time, and calories.

JSON is parsed and validated automatically to ensure clean integration.

**Retrieval-Augmented Generation (RAG)**

Recipes are embedded using models/embedding-001 and stored in a FAISS vector index.

A user query dynamically retrieves similar recipes, which are used to enhance the generated response.

**Image Understanding**

Users can upload food images.

Gemini Vision model is used to analyze the image and describe the dish and how it's prepared.

**GenAI Evaluation**

The assistant can evaluate any recipe JSON for dietary classification (e.g., vegetarian, low-carb) and assign a nutritional score from 1 to 10.

# Step 1: Install the Dependencies

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip uninstall -qqy jupyterlab  # Remove unused conflicting packages
!pip install google-generativeai
!pip install faiss-cpu
!pip install tqdm

import google.generativeai as genai
from google.api_core.retry import Retry
from kaggle_secrets import UserSecretsClient
from PIL import Image
from IPython.display import Markdown, display
from google.genai import types
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import warnings
import logging

# Suppress IPython history warnings
warnings.filterwarnings("ignore", message=".*history saving thread.*")
logging.getLogger('IPython').setLevel(logging.CRITICAL)


In [ ]:
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)


In [ ]:
model = genai.GenerativeModel(model_name="models/gemini-1.5-pro")
vision_model = genai.GenerativeModel(model_name="models/gemini-1.5-flash")

# Step 2: Load and Prepare the Dataset

* A CSV of over 5,000 recipes is loaded and cleaned.

* Each recipe is structured into a single text block for embedding.

In [ ]:
recipes_df = pd.read_csv("/kaggle/input/d/pes12017000148/food-ingredients-and-recipe-dataset-with-images/Food Ingredients and Recipe Dataset with Image Name Mapping.csv")
recipes_df = recipes_df[['Title', 'Ingredients']].dropna().rename(columns={
    'Title': 'name',
    'Ingredients': 'ingredients'
}).head(100)

print(recipes_df.head())

In [ ]:
recipes_df['text'] = recipes_df.apply(lambda row: f"Recipe: {row['name']}\nIngredients: {row['ingredients']}", axis=1)
texts = recipes_df['text'].tolist()

# Step 3: Users Input

* Users enter the ingredients.

* These are turned into queries that drive recipe generation or search.

In [ ]:
user_input = "tomato, onion, chicken"

# Process input
user_ingredients = [ingredient.strip() for ingredient in user_input.split(",") if ingredient.strip()]
ingredient_string = ", ".join(user_ingredients)

# Display formatted output
if user_ingredients:
    md_output = f"### You Entered:\n- " + "\n- ".join(user_ingredients)
else:
    md_output = " No ingredients provided. Please enter at least one."

display(Markdown(md_output))

# Step 4: Structured Recipe Generation

* A prompt to Gemini generates a JSON-formatted recipe based on ingredients.

* Output is validated with json.loads() and errors handled.

In [ ]:
recipe_prompt = f"""
You are an expert AI Chef.

Based on the following ingredients: {ingredient_string}

Generate a creative recipe in valid JSON format with these fields:
- title (string)
- ingredients (list of objects with 'item' and 'amount')
- steps (ordered list of cooking instructions)
- cooking_time (e.g. "30 minutes")
- estimated_calories (number)

Only return valid JSON — no markdown, explanation, or extra notes.
"""

response = model.generate_content(recipe_prompt)
recipe_json = response.text.strip()

if recipe_json.startswith("```json"):
    recipe_json = recipe_json.replace("```json", "").replace("```", "").strip()

try:
    recipe = json.loads(recipe_json)
    print("Recipe generated successfully!")
except json.JSONDecodeError as e:
    print("JSON format error. Here's the raw response:")
    print(recipe_json)
    print("\nParsing error:", e)
    recipe = None

In [ ]:
if recipe:
    # Start building the Markdown string
    md_output = f"### 🍽️ {recipe['title']}\n\n"
    md_output += "**Ingredients:**\n"
    for i in recipe['ingredients']:
        md_output += f"- {i['item']}: {i['amount']}\n"

    md_output += "\n**Steps:**\n"
    for idx, step in enumerate(recipe['steps'], 1):
        md_output += f"{idx}. {step}\n"

    md_output += f"\n**Time:** {recipe['cooking_time']}\n"
    md_output += f"**Calories:** {recipe['estimated_calories']}\n"

    # Display the Markdown
    display(Markdown(md_output))

In [ ]:
user_set = set([i.lower() for i in user_ingredients])
recipe_items = [i['item'].lower() for i in recipe['ingredients']]
missing = list(set(recipe_items) - user_set)

# Build markdown output
if missing:
    md_output = f"### 🛒 Missing Ingredients\n- " + "\n- ".join(missing)
else:
    md_output = "### ✅ You're all set! 👨‍🍳\nNo missing ingredients."

display(Markdown(md_output))

# Step 5: Recipe Evaluation

* Another Gemini prompt analyzes and scores the recipe from a nutritionist’s point of view.
* Uses Markdown to display readable, formatted evaluation.

In [ ]:
eval_prompt = f"""
You are a nutritionist. Review the recipe below and:
- Comment if it's high protein / low carb / vegetarian
- Rate the overall nutrition on a scale of 1 to 10

Recipe:
{json.dumps(recipe, indent=2)}
"""

eval_response = model.generate_content(eval_prompt)
display(Markdown(eval_response.text))


# Step 6: Embedding, Vector Search and RAG

* Recipes are embedded with Google Generative AI's embedding model.

* FAISS is used for fast similarity-based retrieval.

In [ ]:
embeddings = []
for text in tqdm(texts):
    response = genai.embed_content(model="models/embedding-001", content=text, task_type="retrieval_document")
    embeddings.append(response["embedding"])
embedding_array = np.array(embeddings).astype("float32")


In [ ]:
import faiss

index = faiss.IndexFlatL2(embedding_array.shape[1])
index.add(embedding_array)
id_to_text = {i: texts[i] for i in range(len(texts))}

In [ ]:

query = "What can I cook with spinach, chicken, and eggs?"

# Embed and search
query_embedding = genai.embed_content(
    model="models/embedding-001",
    content=query,
    task_type="retrieval_query"
)['embedding']
query_vector = np.array([query_embedding]).astype("float32")

# Retrieve top 5 matches
_, indices = index.search(query_vector, k=5)
retrieved = [id_to_text[i] for i in indices[0]]


In [ ]:
context = "\n".join(retrieved)
chat = model.start_chat()
response = chat.send_message(f"Using the following recipes:\n{context}\n\n{query}")
Markdown(response.text)

# Step 7: Image understanding

* Image files added via "Add Data" are passed to Gemini Vision.

* The model returns dish name and prep steps using vision-language understanding.

In [ ]:
image_path = "/kaggle/input/d/pes12017000148/food-ingredients-and-recipe-dataset-with-images/Food Images/Food Images/-fried-chicken-51238060.jpg"

if os.path.exists(image_path):
    img = Image.open(image_path)
    vision_prompt = "What dish is this and how is it prepared?."
    vision_response = vision_model.generate_content([vision_prompt, img])
    display(Markdown(vision_response.text))
else:
    print("No image uploaded for analysis.")

# Conclusion

**This GenAI Kitchen Assistant project blends creativity, AI reasoning, and user interactivity. It demonstrates a strong use of GenAI in solving a relatable real-world problem while showcasing multiple advanced capabilities.**

* Fully interactive
* Supports structured & unstructured input
* Demonstrates 4+ GenAI capabilities

